# Image Cleanser Pipeline


## Params

In [ ]:
params = {
    "on_Google_drive": True,
    "download_dataset": True,
    "download_model": True,
    "evaluate_model": True,
    "evaluate_dataset": True,
}

## R and R_prime

### Environment

In [ ]:
%pip install parameters fire lmdb pillow torchvision nltk natsort datasets gdown opencv-python scikit-image scipy lpips pyiqa

In [ ]:
# Mount to drive
if not params["on_Google_drive"]:
  print ("Skipping mounting Google Drive")
else:
  from google.colab import drive
  drive.mount('/content/drive')
  import os
  # Change to your own Google Drive repo location
  os.chdir('/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser')

In [ ]:
!pip -q install gdown

FILE_ID = "1Gk6TgEfBoSy3j8Cp05AWgCqU0zqp90"
ZIP = "datasets.zip"

import os, gdown
if not os.path.exists(ZIP):
    gdown.download(id=FILE_ID, output=ZIP, quiet=False)

!unzip -qo datasets.zip -d .

try:
    params["download_dataset"] = False
except:
    pass

!ls -R | head -n 50


In [ ]:
if not params["download_model"]:
  print ("Skipping downloading dataset")
else:
  print ("Use the below commands or manually download and upload")
  print ("Put model files to saved_models/")
  # models = {
  #     'None-ResNet-None-CTC.pth': 'https://drive.google.com/open?id=1FocnxQzFBIjDT2F9BkNUiLdo1cC3eaO0',
  #     'None-VGG-BiLSTM-CTC.pth': 'https://drive.google.com/open?id=1GGC2IRYEMQviZhqQpbtpeTgHO_IXWetG',
  #     'None-VGG-None-CTC.pth': 'https://drive.google.com/open?id=1FS3aZevvLiGF1PFBm5SkwvVcgI6hJWL9',
  #     'TPS-ResNet-BiLSTM-Attn-case-sensitive.pth': 'https://drive.google.com/open?id=1ajONZOgiG9pEYsQ-eBmgkVbMDuHgPCaY',
  #     'TPS-ResNet-BiLSTM-Attn.pth': 'https://drive.google.com/open?id=1b59rXuGGmKne1AuHnkgDzoYgKeETNMv9',
  #     'TPS-ResNet-BiLSTM-CTC.pth': 'https://drive.google.com/open?id=1FocnxQzFBIjDT2F9BkNUiLdo1cC3eaO0',
  # }

  # for k, v in models.items():
  #   doc_id = v[v.find('=')+1:]
  #   !curl -c /tmp/cookies "https://drive.google.com/uc?export=download&id=$doc_id" > /tmp/intermezzo.html
  #   !curl -L -b /tmp/cookies "https://drive.google.com$(cat /tmp/intermezzo.html | grep -Po 'uc-download-link" [^>]* href="\K[^"]*' | sed 's/\&amp;/\&/g')" > $k

  # !ls -al *.pth

### Helper Functions

In [ ]:

def create_lmdb(image_directory_original, gt_file_original, lmdb_output_dir):
    # Create the output directory if it doesn't exist
    os.makedirs(lmdb_output_dir, exist_ok=False)


    # Ensure the create_lmdb_dataset.py script exists in your current directory
    if os.path.exists('/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/deep-text-recognition-benchmark/create_lmdb_dataset.py'):
        print(f"Creating LMDB dataset from {image_directory_original} to {lmdb_output_dir} using {gt_file_original}...")
        # The script create_lmdb_dataset.py expects inputPath, gtFile, and outputPath positionally
        create_lmdb_command = f'python3 /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/deep-text-recognition-benchmark/create_lmdb_dataset.py {image_directory_original} {gt_file_original} {lmdb_output_dir}'
        !{create_lmdb_command}
        print("LMDB dataset creation finished.")
    else:
        print("Error: create_lmdb_dataset.py not found. Please make sure it's in the current directory.")

In [ ]:
# Scorers
import cv2
import numpy as np
from skimage.morphology import skeletonize

def normalize_score(x, min_val=0, max_val=1):
    """Clamp and normalize to [0,1]."""
    return float(np.clip((x - min_val) / (max_val - min_val + 1e-8), 0, 1))


def calculate_sharp_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img.copy()

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    tenengrad_score = float(np.mean(gx**2 + gy**2))

    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    mag = np.abs(fshift)
    h, w = mag.shape
    c = 10
    center = mag[h//2 - c:h//2 + c, w//2 - c:w//2 + c]
    high = mag.sum() - center.sum()
    fft_ratio = float(high / (mag.sum() + 1e-8))

    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.mean(bw) > 127:
        bw = 255 - bw
    skel = skeletonize(bw // 255).astype(np.uint8)
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(skel, connectivity=8)
    if num_labels > 1:
        lengths = stats[1:, cv2.CC_STAT_AREA]
        continuity = float(np.mean(lengths) / (np.std(lengths) + 1e-6))
    else:
        continuity = 0.0

    # normalize sub-scores
    t_norm = np.tanh(np.log1p(tenengrad_score) / 5.0)
    f_norm = normalize_score(fft_ratio, 0, 1)
    c_norm = np.tanh(continuity / 10.0)

    combined = 0.6 * t_norm + 0.3 * f_norm + 0.1 * c_norm
    return normalize_score(combined, 0, 1)


def calculate_low_light_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img.copy()
    meanY = gray.mean()
    dark_ratio = np.mean(gray < 50)
    alpha, beta = 0.7, 0.3
    score = alpha * (1 - meanY / 255.0) + beta * dark_ratio
    return normalize_score(score, 0, 1)


def calculate_contrast_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img.copy()
    p10 = np.percentile(gray, 10)
    p90 = np.percentile(gray, 90)
    contrast = (p90 - p10) / 255.0
    return normalize_score(contrast, 0, 1)


def calculate_noise_score(img, edge_exclude=0.20, ref_sigma=12.0):
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img.copy()

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx * gx + gy * gy)

    if edge_exclude > 0:
        thr = np.percentile(mag, 100 * (1 - edge_exclude))
        flat_mask = (mag < thr)
    else:
        flat_mask = np.ones_like(gray, dtype=bool)

    g = gray.astype(np.float32)
    flat_mask &= (g > 5) & (g < 250)
    if flat_mask.mean() < 0.10:
        flat_mask = np.ones_like(gray, dtype=bool)

    hp = g - cv2.GaussianBlur(g, (0, 0), 1.0)
    hp_flat = hp[flat_mask]
    mad = np.median(np.abs(hp_flat - np.median(hp_flat)))
    sigma_mad = 1.4826 * mad

    ker = np.array([[1, -2, 1],
                    [-2, 4, -2],
                    [1, -2, 1]], dtype=np.float32)
    L = cv2.filter2D(g, cv2.CV_32F, ker)[1:-1, 1:-1]
    L_use = np.abs(L[flat_mask[1:-1, 1:-1]])
    sigma_immer = (np.sqrt(np.pi / 2.0) / 6.0) * np.mean(L_use) if L_use.size > 0 else sigma_mad

    sigma_est = float(np.median([sigma_mad, sigma_immer]))

    # normalize noise: low noise=0, high noise=1
    norm_noise = sigma_est / (ref_sigma + sigma_est)
    return normalize_score(norm_noise, 0, 1)


In [ ]:
# Computing Scorers
import os
import re
import cv2
import pandas as pd
from tqdm import tqdm

def compute_quality_scores(img_dir, out_csv):
    """
    Compute image quality metrics (sharp, noise, contrast, low-light)
    for all images in a folder and save them as a CSV.

    Args:
        img_dir (str): Path to directory with images.
        out_csv (str): Output CSV path.
    """
    # === Ensure output folder exists ===
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)

    # === Helper: natural numeric sort ===
    def natural_key(name):
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", name)]

    # === Collect and sort image files ===
    images = sorted(
        [f for f in os.listdir(img_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))],
        key=natural_key
    )
    print(f"📁 Found {len(images)} images in {img_dir}")

    rows = []
    for fname in tqdm(images, desc=f"Analyzing {os.path.basename(img_dir)}"):
        img_path = os.path.join(img_dir, fname)
        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️ Skipped unreadable image: {fname}")
            continue

        sharp = calculate_sharp_score(img)
        low_light = calculate_low_light_score(img)
        contrast = calculate_contrast_score(img)
        noise = calculate_noise_score(img)

        # assume filename starts with "idx_label.png"
        try:
            idx = int(os.path.splitext(fname)[0].split("_")[0])
        except ValueError:
            print(f"⚠️ Skipped malformed filename: {fname}")
            continue

        rows.append({
            "idx": idx,
            "filename": fname,
            "sharp_score": sharp,
            "low_light_score": low_light,
            "contrast_score": contrast,
            "noise_score": noise,
        })

    # === Save to CSV ===
    df = pd.DataFrame(rows)
    df = df.sort_values("idx").reset_index(drop=True)
    df.to_csv(out_csv, index=False)
    print(f"✅ Saved quality scores for {len(df)} images → {out_csv}")

    return df


In [ ]:
import pandas as pd
import os

def combine_results_and_scores(detailed_csv, score_csv, out_dir=None, tag=None):
    """
    Combine recognition results and quality scores CSVs by `idx`,
    save both full and NED<1 filtered versions, and return the merged DataFrames.

    Args:
        detailed_csv (str): Path to detailed results CSV (from test.py).
        score_csv (str): Path to image quality scores CSV.
        out_dir (str): Directory to save combined CSVs (default = same as detailed_csv).
        tag (str): Optional dataset name for output filenames.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]:
            (merged_full, merged_filtered)
    """
    df_results = pd.read_csv(detailed_csv)
    df_scores = pd.read_csv(score_csv)

    df_results["idx"] = df_results["idx"].astype(int)
    df_scores["idx"] = df_scores["idx"].astype(int)

    merged = pd.merge(df_results, df_scores, on="idx", how="inner")
    print(f"✅ Combined {len(merged)} records from {os.path.basename(detailed_csv)} and {os.path.basename(score_csv)}")

    if out_dir is None:
        out_dir = os.path.dirname(detailed_csv)
    os.makedirs(out_dir, exist_ok=True)
    if tag is None:
        tag = os.path.basename(detailed_csv).replace("_detailed_results.csv", "")

    out_full = os.path.join(out_dir, f"{tag}_combined.csv")
    out_filtered = os.path.join(out_dir, f"{tag}_combined_ned_lt1.csv")

    merged.to_csv(out_full, index=False)
    merged_filtered = merged[merged["norm_edit_distance"] < 1.0].copy()
    merged_filtered.to_csv(out_filtered, index=False)

    print(f"💾 Saved: {out_full}\n💾 Saved filtered (<1.0 NED): {out_filtered}")

    return merged, merged_filtered


### Evaluating Model and Datasets
- M: None-VGG-None-CTC.pth
- D: MJ600 - 600 pictures out of MJSynth datasets which was used to train M
- D': D_prime - 400 pictures out of IC13 which is a real-life text dataset, plus 200 pictures taken by phone which are intentionally blurred text pictures

In [ ]:
if not params["evaluate_model"]:
  print ("Skipping evaluating model")
else:
  # Evaluate model on D_prime
  !python3 /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/deep-text-recognition-benchmark/test.py \
  --eval_data /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime/D_prime_lmdb \
  --data_filtering_off \
  --Transformation None --FeatureExtraction VGG --SequenceModeling None --Prediction CTC \
  --saved_model /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/deep-text-recognition-benchmark/modules/None-VGG-None-CTC.pth

In [ ]:
if not params["evaluate_model"]:
  print ("Skipping evaluating model")
else:
  # Evaluate model on MJ600
  !python3 /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/deep-text-recognition-benchmark/test.py \
  --eval_data /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/MJ600/MJ600_lmdb \
  --data_filtering_off \
  --Transformation None --FeatureExtraction VGG --SequenceModeling None --Prediction CTC \
  --saved_model /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/deep-text-recognition-benchmark/modules/None-VGG-None-CTC.pth

In [ ]:
if not params["evaluate_dataset"]:
  print ("Skipping evaluating dataset")
else:
  # Compute scorers for the datasets
  compute_quality_scores("/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime/image","./result/None-VGG-None-CTC.pth/D_prime_quality_scores.csv")

In [ ]:
if not params["evaluate_dataset"]:
  print ("Skipping evaluating dataset")
else:
  compute_quality_scores("/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/MJ600/image","./result/None-VGG-None-CTC.pth/MJ600_quality_scores.csv")

In [ ]:
if not params["evaluate_dataset"]:
  print ("Skipping evaluating dataset")
else:
  # Combine model evaluation and dataset score for better visualization
  combine_results_and_scores("/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/result/content_drive_MyDrive_UCSD_COURSES_ECE253_ImageCleanser_deep-text-recognition-benchmark_modules_None-VGG-None-CTC.pth/D_prime_detailed_results.csv", "./result/None-VGG-None-CTC.pth/D_prime_quality_scores.csv", "./result/None-VGG-None-CTC.pth/", "D_prime")

In [ ]:
if not params["evaluate_dataset"]:
  print ("Skipping evaluating dataset")
else:
  # Combine model evaluation and dataset score for better visualization
  combine_results_and_scores("/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/result/content_drive_MyDrive_UCSD_COURSES_ECE253_ImageCleanser_deep-text-recognition-benchmark_modules_None-VGG-None-CTC.pth/MJ600_detailed_results.csv", "./result/None-VGG-None-CTC.pth/MJ600_quality_scores.csv", "./result/None-VGG-None-CTC.pth/", "MJ600")

### Visualization
- Model performance on datasets is evaluated by distribution of confidence and normalized edit distance
- Image quality in a datset is evaluated by distribution of sharp_score, noise_score, constrast_score, and low_light_score
- Correlation is shown by mapping model performance metrics to image quality

#### Model Performance

In [ ]:
!cd /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df1 = pd.read_csv("/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/result/content_drive_MyDrive_UCSD_COURSES_ECE253_ImageCleanser_deep-text-recognition-benchmark_modules_None-VGG-None-CTC.pth/MJ600_detailed_results.csv")
df2 = pd.read_csv("/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/result/content_drive_MyDrive_UCSD_COURSES_ECE253_ImageCleanser_deep-text-recognition-benchmark_modules_None-VGG-None-CTC.pth/D_prime_detailed_results.csv")

df1["dataset"] = "MJ600"
df2["dataset"] = "D′"

df_all = pd.concat([df1, df2], ignore_index=True)

plt.figure(figsize=(8, 4))
plt.hist(df1["confidence"], bins=30, alpha=0.5, label="MJ600", density=True)
plt.hist(df2["confidence"], bins=30, alpha=0.5, label="D′", density=True)
plt.title("Distribution of Confidence Scores")
plt.xlabel("Confidence")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(df1["norm_edit_distance"], bins=30, alpha=0.5, label="MJ600", density=True)
plt.hist(df2["norm_edit_distance"], bins=30, alpha=0.5, label="D′", density=True)
plt.title("Distribution of Normalized Edit Distance")
plt.xlabel("Normalized Edit Distance")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.show()

summary = df_all.groupby("dataset")[["confidence", "norm_edit_distance"]].describe()
display(summary)


In [ ]:
import pandas as pd
import os

df1 = pd.read_csv("./result/None-VGG-None-CTC.pth/MJ600_combined_ned_lt1.csv")
df2 = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_combined_ned_lt1.csv")

df1_filtered_confidence = df1[(df1['confidence'] > 0) & (df1['confidence'] < 1)].copy()
df2_filtered_confidence = df2[(df2['confidence'] > 0) & (df2['confidence'] < 1)].copy()

output_dir = "./result/None-VGG-None-CTC.pth/"
os.makedirs(output_dir, exist_ok=True) # Ensure output directory exists

output_path_mj600 = os.path.join(output_dir, "MJ600_combined_ned_lt1_conf_gt0lt1.csv")
output_path_dprime = os.path.join(output_dir, "D_prime_combined_ned_lt1_conf_gt0lt1.csv")

# Save the filtered dataframes to new CSV files
df1_filtered_confidence.to_csv(output_path_mj600, index=False)
df2_filtered_confidence.to_csv(output_path_dprime, index=False)

print(f"Saved filtered MJ600 data (NED < 1, Confidence > 0 and < 1) to: {output_path_mj600}")
print(f"Saved filtered D_prime data (NED < 1, Confidence > 0 and < 1) to: {output_path_dprime}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df1 = pd.read_csv("./result/None-VGG-None-CTC.pth/MJ600_combined_ned_lt1_conf_gt0lt1.csv")
df2 = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_combined_ned_lt1_conf_gt0lt1.csv")

df1["dataset"] = "MJ600"
df2["dataset"] = "D′"

df_all = pd.concat([df1, df2], ignore_index=True)

plt.figure(figsize=(8, 4))
plt.hist(df1["confidence"], bins=30, alpha=0.5, label="MJ600", density=True)
plt.hist(df2["confidence"], bins=30, alpha=0.5, label="D′", density=True)
plt.title("Distribution of Confidence Scores")
plt.xlabel("Confidence")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(df1["norm_edit_distance"], bins=30, alpha=0.5, label="MJ600", density=True)
plt.hist(df2["norm_edit_distance"], bins=30, alpha=0.5, label="D′", density=True)
plt.title("Distribution of Normalized Edit Distance")
plt.xlabel("Normalized Edit Distance")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.show()

summary = df_all.groupby("dataset")[["confidence", "norm_edit_distance"]].describe()
display(summary)


#### Image List and Metrics

In [ ]:
# D_prime
from IPython.display import display, HTML
from PIL import Image
import pandas as pd
import io, base64, os

csv_path = "./result/None-VGG-None-CTC.pth/D_prime_combined.csv"
img_dir  = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime/image"

df = pd.read_csv(csv_path)

rows = []
missing = []

for _, row in df.iterrows():
    idx = int(row["idx"])  # filenames start from 1
    gt   = str(row["ground_truth"])
    pred = str(row["prediction"])
    conf = row["confidence"]
    ned  = row["norm_edit_distance"]

    # match by index prefix only
    candidates = [f for f in os.listdir(img_dir) if f.startswith(f"{idx}_")]
    if not candidates:
        missing.append(idx)
        continue
    img_name = candidates[0]
    img_path = os.path.join(img_dir, img_name)

    img = Image.open(img_path)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    rows.append({
        "idx": idx,
        "image": f'<img src="data:image/png;base64,{img_b64}" width="160">',
        "filename": img_name,
        "ground_truth": gt,
        "prediction": pred,
        "confidence": f"{conf:.4f}",
        "norm_edit_distance": f"{ned:.4f}",
        "sharp_score":  f"{row['sharp_score']:.3f}",
        "noise_score": f"{row['noise_score']:.3f}",
        "contrast_score": f"{row['contrast_score']:.3f}",
        "low_light_score": f"{row['low_light_score']:.3f}",
    })

display_df = pd.DataFrame(rows)

# --- scrollable HTML output ---
html_table = display_df.to_html(escape=False, index=False)
scrollable_html = f"""
<div style="
    max-height: 600px;
    overflow-y: auto;
    overflow-x: auto;
    border: 1px solid #ccc;
    padding: 8px;
">
{html_table}
</div>
"""
display(HTML(scrollable_html))

if missing:
    print(f"⚠️ Missing {len(missing)} images: {missing[:10]}")


In [ ]:
# MJ600
from IPython.display import display, HTML
from PIL import Image
import pandas as pd
import io, base64, os

csv_path = "./result/None-VGG-None-CTC.pth/MJ600_combined.csv"
img_dir  = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/MJ600/image"

df = pd.read_csv(csv_path)

rows = []
missing = []

for _, row in df.iterrows():
    idx = int(row["idx"]) # filenames start from 1
    gt   = str(row["ground_truth"])
    pred = str(row["prediction"])
    conf = row["confidence"]
    ned  = row["norm_edit_distance"]

    # match by index prefix only
    candidates = [f for f in os.listdir(img_dir) if f.startswith(f"{idx}_")]
    if not candidates:
        missing.append(idx)
        continue
    img_name = candidates[0]
    img_path = os.path.join(img_dir, img_name)

    img = Image.open(img_path)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    rows.append({
        "idx": idx,
        "image": f'<img src="data:image/png;base64,{img_b64}" width="160">',
        "filename": img_name,
        "ground_truth": gt,
        "prediction": pred,
        "confidence": f"{conf:.4f}",
        "norm_edit_distance": f"{ned:.4f}",
        "sharp_score":  f"{row['sharp_score']:.3f}",
        "noise_score": f"{row['noise_score']:.3f}",
        "contrast_score": f"{row['contrast_score']:.3f}",
        "low_light_score": f"{row['low_light_score']:.3f}",
    })

display_df = pd.DataFrame(rows)

html_table = display_df.to_html(escape=False, index=False)
scrollable_html = f"""
<div style="
    max-height: 600px;
    overflow-y: auto;
    overflow-x: auto;
    border: 1px solid #ccc;
    padding: 8px;
">
{html_table}
</div>
"""
display(HTML(scrollable_html))

if missing:
    print(f"⚠️ Missing {len(missing)} images: {missing[:10]}")


#### Image List and Metrics for NED < 1

In [ ]:
# D_prime
from IPython.display import display, HTML
from PIL import Image
import pandas as pd
import io, base64, os

csv_path = "./result/None-VGG-None-CTC.pth/D_prime_combined_ned_lt1.csv"
img_dir  = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime/image"

df = pd.read_csv(csv_path)

rows = []
missing = []

for _, row in df.iterrows():
    idx = int(row["idx"])  # filenames start from 1
    gt   = str(row["ground_truth"])
    pred = str(row["prediction"])
    conf = row["confidence"]
    ned  = row["norm_edit_distance"]

    # match by index prefix only
    candidates = [f for f in os.listdir(img_dir) if f.startswith(f"{idx}_")]
    if not candidates:
        missing.append(idx)
        continue
    img_name = candidates[0]
    img_path = os.path.join(img_dir, img_name)

    img = Image.open(img_path)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    rows.append({
        "idx": idx,
        "image": f'<img src="data:image/png;base64,{img_b64}" width="160">',
        "filename": img_name,
        "ground_truth": gt,
        "prediction": pred,
        "confidence": f"{conf:.4f}",
        "norm_edit_distance": f"{ned:.4f}",
        "sharp_score":  f"{row['sharp_score']:.3f}",
        "noise_score": f"{row['noise_score']:.3f}",
        "contrast_score": f"{row['contrast_score']:.3f}",
        "low_light_score": f"{row['low_light_score']:.3f}",
    })

display_df = pd.DataFrame(rows)

# --- scrollable HTML output ---
html_table = display_df.to_html(escape=False, index=False)
scrollable_html = f"""
<div style="
    max-height: 600px;
    overflow-y: auto;
    overflow-x: auto;
    border: 1px solid #ccc;
    padding: 8px;
">
{html_table}
</div>
"""
display(HTML(scrollable_html))

if missing:
    print(f"⚠️ Missing {len(missing)} images: {missing[:10]}")


In [ ]:
# D_prime
from IPython.display import display, HTML
from PIL import Image
import pandas as pd
import io, base64, os

csv_path = "result/None-VGG-None-CTC.pth/MJ600_combined_ned_lt1.csv"
img_dir  = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/MJ600/image"

df = pd.read_csv(csv_path)

rows = []
missing = []

for _, row in df.iterrows():
    idx = int(row["idx"])
    gt   = str(row["ground_truth"])
    pred = str(row["prediction"])
    conf = row["confidence"]
    ned  = row["norm_edit_distance"]

    # match by index prefix only
    candidates = [f for f in os.listdir(img_dir) if f.startswith(f"{idx}_")]
    if not candidates:
        missing.append(idx)
        continue
    img_name = candidates[0]
    img_path = os.path.join(img_dir, img_name)

    img = Image.open(img_path)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    rows.append({
        "idx": idx,
        "image": f'<img src="data:image/png;base64,{img_b64}" width="160">',
        "filename": img_name,
        "ground_truth": gt,
        "prediction": pred,
        "confidence": f"{conf:.4f}",
        "norm_edit_distance": f"{ned:.4f}",
        "sharp_score":  f"{row['sharp_score']:.3f}",
        "noise_score": f"{row['noise_score']:.3f}",
        "contrast_score": f"{row['contrast_score']:.3f}",
        "low_light_score": f"{row['low_light_score']:.3f}",
    })

display_df = pd.DataFrame(rows)

html_table = display_df.to_html(escape=False, index=False)
scrollable_html = f"""
<div style="
    max-height: 600px;
    overflow-y: auto;
    overflow-x: auto;
    border: 1px solid #ccc;
    padding: 8px;
">
{html_table}
</div>
"""
display(HTML(scrollable_html))

if missing:
    print(f"⚠️ Missing {len(missing)} images: {missing[:10]}")


#### Image Scorer Distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

csv_dprime = "./result/None-VGG-None-CTC.pth/D_prime_combined.csv"
csv_mj600  = "./result/None-VGG-None-CTC.pth/MJ600_combined.csv"

df_dprime = pd.read_csv(csv_dprime)
df_mj600  = pd.read_csv(csv_mj600)

df_dprime["dataset"] = "D_prime"
df_mj600["dataset"]  = "MJ600"

df_all = pd.concat([df_dprime, df_mj600], ignore_index=True)

metrics = ["sharp_score", "noise_score", "contrast_score", "low_light_score"]
titles  = ["Sharp", "Noise", "Contrast", "Low-light"]

sns.set(style="whitegrid")

for m, t in zip(metrics, titles):
    plt.figure(figsize=(8, 4))
    sns.kdeplot(
        data=df_all, x=m, hue="dataset", fill=True, common_norm=False,
        alpha=0.5, linewidth=1.2
    )
    plt.title(f"{t} Score Distribution: D_prime vs MJ600")
    plt.xlabel("Normalized score (0–1)")
    plt.ylabel("Density")
    plt.grid(True)
    plt.show()

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

def compute_blur_variance(image_path):

    img = cv2.imread(image_path)
    if img is None:
        return 0

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img


    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    variance = laplacian.var()

    return variance

def categorize_blur_levels(image_dir, output_csv):

    image_files = sorted([f for f in os.listdir(image_dir)
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    results = []
    for fname in tqdm(image_files, desc="Computing blur variance"):
        img_path = os.path.join(image_dir, fname)
        variance = compute_blur_variance(img_path)

        try:
            idx = int(fname.split('_')[0])
        except:
            idx = -1
        if variance > 100:
            blur_level = 'light'
        elif variance > 50:
            blur_level = 'moderate'
        else:
            blur_level = 'heavy'

        results.append({
            'idx': idx,
            'filename': fname,
            'blur_variance': variance,
            'blur_level': blur_level
        })

    df = pd.DataFrame(results)
    df = df.sort_values('idx').reset_index(drop=True)
    df.to_csv(output_csv, index=False)

    print("\n" + "="*60)
    print("Blur Level Distribution:")
    print("="*60)
    print(df['blur_level'].value_counts())
    print(f"\nMean variance by level:")
    print(df.groupby('blur_level')['blur_variance'].mean())

    return df

df_blur = categorize_blur_levels(
    image_dir="/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime/image",
    output_csv="./result/None-VGG-None-CTC.pth/D_prime_blur_levels.csv"
)

print(f"\n✅ Total images classified: {len(df_blur)}")


In [ ]:
def apply_usm_deblur(img, k=1.5, sigma=1.0):

    if img is None:
        return None

    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)

    blurred = cv2.GaussianBlur(img, (0, 0), sigma)
    sharpened = cv2.addWeighted(img, 1 + k, blurred, -k, 0)
    sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)
    return sharpened




def generate_usm_enhanced_dataset(
    input_dir,
    output_dir,
    k=1.5,
    sigma=1.0
):

    os.makedirs(output_dir, exist_ok=True)

    image_files = sorted([f for f in os.listdir(input_dir)
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    print(f"\n{'='*60}")
    print(f"Applying USM Deblurring (k={k}, sigma={sigma})")
    print(f"{'='*60}")
    print(f"Input:  {input_dir}")
    print(f"Output: {output_dir}")
    print(f"Total images: {len(image_files)}")


    processing_log = []
    for fname in tqdm(image_files, desc="Processing with USM"):
        img_path = os.path.join(input_dir, fname)
        img = cv2.imread(img_path)

        if img is None:
            print(f"⚠️ Skipped unreadable image: {fname}")
            continue
        enhanced = apply_usm_deblur(img, k=k, sigma=sigma)

        out_path = os.path.join(output_dir, fname)
        cv2.imwrite(out_path, enhanced)

        processing_log.append({
            'filename': fname,
            'method': 'USM',
            'k': k,
            'sigma': sigma
        })

    print(f"\n✅ Successfully processed {len(processing_log)} images")

    return pd.DataFrame(processing_log)

print("\n" + "="*70)
print("EXPERIMENT: USM Deblurring on D_prime")
print("="*70)

df_usm_log = generate_usm_enhanced_dataset(
    input_dir="/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime/image",
    output_dir="/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime_enhanced_USM/image",
    k=1.5,
    sigma=1.0
)

df_usm_log.to_csv("./result/None-VGG-None-CTC.pth/USM_processing_log.csv", index=False)
print(f"\n📝 Processing log saved to: ./result/None-VGG-None-CTC.pth/USM_processing_log.csv")


In [ ]:
import cv2
import numpy as np
import time
from scipy import ndimage, signal
from scipy.optimize import minimize_scalar
from skimage import restoration
from skimage.restoration import richardson_lucy

def apply_wiener_deblur(img, psf_size=15, noise_var=0.01):

    if img.dtype != np.float64:
        img_float = img.astype(np.float64)
    else:
        img_float = img.copy()
    psf = np.zeros((psf_size, psf_size), dtype=np.float64)
    center = psf_size // 2
    y, x = np.ogrid[:psf_size, :psf_size]
    mask = (x - center)**2 + (y - center)**2 <= (psf_size//3)**2
    psf[mask] = 1
    psf = psf / psf.sum()

    if img.ndim == 3:
        deblurred = np.zeros_like(img_float)
        for i in range(3):
            channel = img_float[:, :, i]
            deblurred_channel = restoration.wiener(channel, psf, noise_var, clip=False)
            deblurred[:, :, i] = deblurred_channel
    else:
        deblurred = restoration.wiener(img_float, psf, noise_var, clip=False)

    deblurred = np.clip(deblurred, 0, 255).astype(np.uint8)
    return deblurred


def apply_richardson_lucy_deblur(img, psf_size=9, iterations=30):
    if img.dtype != np.float64:
        img_float = img.astype(np.float64)
    else:
        img_float = img.copy()

    psf = np.zeros((psf_size, psf_size), dtype=np.float64)
    psf[psf_size//2, :] = 1.0 / psf_size

    if img.ndim == 3:
        deblurred = np.zeros_like(img_float)
        for i in range(3):
            channel = img_float[:, :, i]
            deblurred_channel = richardson_lucy(channel, psf, num_iter=iterations, clip=False)
            deblurred[:, :, i] = deblurred_channel
    else:
        deblurred = richardson_lucy(img_float, psf, num_iter=iterations, clip=False)

    deblurred = np.clip(deblurred, 0, 255).astype(np.uint8)
    return deblurred


def estimate_blur_kernel(img, kernel_size=15):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float64)
    center = kernel_size // 2
    kernel[center, :] = 1.0 / kernel_size

    return kernel


def apply_blind_deblur(img, max_iter=5, kernel_size=15):
    if img.dtype != np.float64:
        img_float = img.astype(np.float64)
    else:
        img_float = img.copy()
    psf = estimate_blur_kernel(img, kernel_size)

    if img.ndim == 3:
        deblurred = np.zeros_like(img_float)
        for i in range(3):
            channel = img_float[:, :, i]
            deblurred_channel = channel.copy()
            for iter_idx in range(max_iter):
                # Deconvolve with current kernel estimate
                deblurred_channel = richardson_lucy(deblurred_channel, psf, num_iter=10, clip=False)
            deblurred[:, :, i] = deblurred_channel
    else:
        deblurred = img_float.copy()
        for iter_idx in range(max_iter):
            deblurred = richardson_lucy(deblurred, psf, num_iter=10, clip=False)
    deblurred = np.clip(deblurred, 0, 255).astype(np.uint8)
    return deblurred


def generate_deblurred_dataset(input_dir, output_dir, method='USM', **kwargs):
    os.makedirs(output_dir, exist_ok=True)

    image_files = sorted([f for f in os.listdir(input_dir)
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    print(f"\n{'='*60}")
    print(f"Applying {method} Deblurring")
    print(f"{'='*60}")
    print(f"Input:  {input_dir}")
    print(f"Output: {output_dir}")
    print(f"Total images: {len(image_files)}")

    processing_log = []

    for fname in tqdm(image_files, desc=f"Processing with {method}"):
        img_path = os.path.join(input_dir, fname)
        img = cv2.imread(img_path)

        if img is None:
            print(f"⚠️ Skipped unreadable image: {fname}")
            continue

        # Measure processing time
        start_time = time.time()

        # Apply deblurring method
        try:
            if method == 'USM':
                enhanced = apply_usm_deblur(img,
                                          k=kwargs.get('k', 1.5),
                                          sigma=kwargs.get('sigma', 1.0))
            elif method == 'Wiener':
                enhanced = apply_wiener_deblur(img,
                                             psf_size=kwargs.get('psf_size', 15),
                                             noise_var=kwargs.get('noise_var', 0.01))
            elif method == 'RL':
                enhanced = apply_richardson_lucy_deblur(img,
                                                       psf_size=kwargs.get('psf_size', 9),
                                                       iterations=kwargs.get('iterations', 30))
            elif method == 'Blind':
                enhanced = apply_blind_deblur(img,
                                           max_iter=kwargs.get('max_iter', 5),
                                           kernel_size=kwargs.get('kernel_size', 15))
            else:
                raise ValueError(f"Unknown method: {method}")
        except Exception as e:
            print(f"⚠️ Error processing {fname}: {e}")
            continue

        processing_time = time.time() - start_time

        # Save
        out_path = os.path.join(output_dir, fname)
        cv2.imwrite(out_path, enhanced)
        try:
            idx = int(fname.split('_')[0])
        except:
            idx = -1
        log_entry = {
            'idx': idx,
            'filename': fname,
            'method': method,
            'processing_time': processing_time
        }
        log_entry.update(kwargs)
        processing_log.append(log_entry)

    print(f"\n✅ Successfully processed {len(processing_log)} images")

    return pd.DataFrame(processing_log)

print("✅ Deblurring functions defined (Wiener, RL, Blind)")


In [ ]:

import cv2
import numpy as np

try:
    import torch
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision"])
    import torch

try:
    import lpips
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lpips"])
    import lpips

try:
    from pyiqa import create_metric
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyiqa"])
    from pyiqa import create_metric
_lpips_model = None
_niqe_metric = None

def _init_quality_models():
    global _lpips_model, _niqe_metric
    if _lpips_model is None:
        try:
            _lpips_model = lpips.LPIPS(net="alex")
            _lpips_model.eval()
        except Exception as e:
            print(f"⚠️ LPIPS init failed: {e}")
            _lpips_model = None

    if _niqe_metric is None:
        try:
            _niqe_metric = create_metric("niqe", device="cpu")
        except Exception as e:
            print(f"⚠️ NIQE init failed: {e}")
            _niqe_metric = None

_init_quality_models()
print(f"✅ Quality models ready | LPIPS: {_lpips_model is not None} | NIQE: {_niqe_metric is not None}")

def compute_niqe_score(img):
    if _niqe_metric is None:
        return None
    try:
        if img is None:
            return None
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).unsqueeze(0).float() / 255.0
        score = _niqe_metric(img_tensor)
        return float(score.item())
    except Exception as e:
        print(f"⚠️ NIQE computation error: {e}")
        return None

def compute_lpips_score(img1, img2):
    if _lpips_model is None:
        return None
    try:
        if img1 is None or img2 is None:
            return None
        if img1.ndim == 2:
            img1 = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)
        if img2.ndim == 2:
            img2 = cv2.cvtColor(img2, cv2.COLOR_GRAY2BGR)

        img1_rgb = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        img2_rgb = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

        # [0,1] -> [-1,1]
        img1_t = torch.from_numpy(img1_rgb).permute(2, 0, 1).unsqueeze(0) * 2.0 - 1.0
        img2_t = torch.from_numpy(img2_rgb).permute(2, 0, 1).unsqueeze(0) * 2.0 - 1.0

        with torch.no_grad():
            dist = _lpips_model(img1_t, img2_t)
        return float(dist.item())
    except Exception as e:
        print(f"⚠️ LPIPS computation error: {e}")
        return None


In [ ]:
base_input_dir = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime/image"
base_output_dir = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser"

deblurring_methods = {
    'USM': {
        'output_dir': f"{base_output_dir}/D_prime_enhanced_USM/image",
        'params': {'k': 1.5, 'sigma': 1.0}
    },
    'Wiener': {
        'output_dir': f"{base_output_dir}/D_prime_enhanced_Wiener/image",
        'params': {'psf_size': 15, 'noise_var': 0.01}
    },
    'RL': {
        'output_dir': f"{base_output_dir}/D_prime_enhanced_RL/image",
        'params': {'psf_size': 9, 'iterations': 30}
    },
    'Blind': {
        'output_dir': f"{base_output_dir}/D_prime_enhanced_Blind/image",
        'params': {'max_iter': 5, 'kernel_size': 15}
    }
}

# Process each method
all_processing_logs = {}

print("\n" + "="*70)
print("GENERATING ALL DEBLURRING METHODS")
print("="*70)

for method_name, config in deblurring_methods.items():
    print(f"\n{'='*70}")
    print(f"Processing Method: {method_name}")
    print(f"{'='*70}")

    try:
        log_df = generate_deblurred_dataset(
            input_dir=base_input_dir,
            output_dir=config['output_dir'],
            method=method_name,
            **config['params']
        )

        # Save processing log
        log_path = f"./result/None-VGG-None-CTC.pth/{method_name}_processing_log.csv"
        log_df.to_csv(log_path, index=False)
        all_processing_logs[method_name] = log_df

        print(f"✅ {method_name} processing complete")
        print(f"   Average processing time: {log_df['processing_time'].mean():.4f}s per image")
        print(f"   Log saved to: {log_path}")

    except Exception as e:
        print(f"❌ Error processing {method_name}: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*70)
print("✅ ALL DEBLURRING METHODS PROCESSED")
print("="*70)


In [ ]:
import shutil
import glob
method_configs = {
    'USM': 'D_prime_enhanced_USM',
    'Wiener': 'D_prime_enhanced_Wiener',
    'RL': 'D_prime_enhanced_RL',
    'Blind': 'D_prime_enhanced_Blind'
}

base_path = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser"
model_path = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/deep-text-recognition-benchmark/modules/None-VGG-None-CTC.pth"

print("\n" + "="*70)
print("CREATING LMDB AND EVALUATING OCR FOR ALL METHODS")
print("="*70)

for method_name, dataset_name in method_configs.items():
    print(f"\n{'='*70}")
    print(f"Processing: {method_name} ({dataset_name})")
    print(f"{'='*70}")
    source_gt = f"{base_path}/D_prime/gt.txt"
    target_gt = f"{base_path}/{dataset_name}/gt.txt"

    if not os.path.exists(target_gt):
        os.makedirs(os.path.dirname(target_gt), exist_ok=True)
        shutil.copy(source_gt, target_gt)
        print(f"✅ Copied gt.txt to {dataset_name}")
    image_dir = f"{base_path}/{dataset_name}/image"
    gt_file = f"{base_path}/{dataset_name}/gt.txt"
    lmdb_output = f"{base_path}/{dataset_name}/{dataset_name}_lmdb"

    if os.path.exists(lmdb_output):
        print(f"ℹ️ LMDB already exists: {lmdb_output}")
    else:
        print(f"Creating LMDB for {method_name}...")
        create_lmdb(image_dir, gt_file, lmdb_output)
        print(f"✅ LMDB created: {lmdb_output}")
    result_pattern = f"./result/content_drive_MyDrive_UCSD_COURSES_ECE253_ImageCleanser_deep-text-recognition-benchmark_modules_None-VGG-None-CTC.pth/{dataset_name}_detailed_results.csv"
    existing_results = glob.glob(result_pattern)

    if existing_results:
        print(f"ℹ️ OCR results already exist for {method_name}")
    else:
        print(f"Evaluating OCR model on {method_name}...")
        eval_cmd = f"""python3 /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/deep-text-recognition-benchmark/test.py \\
  --eval_data {lmdb_output} \\
  --data_filtering_off \\
  --Transformation None --FeatureExtraction VGG --SequenceModeling None --Prediction CTC \\
  --saved_model {model_path}"""

        !{eval_cmd}
        print(f"✅ OCR evaluation complete for {method_name}")

print("\n" + "="*70)
print("✅ ALL METHODS EVALUATED")
print("="*70)


In [ ]:
import torch
import cv2
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import pyiqa

def compute_quality_metrics_batch(original_dir, enhanced_dir, output_csv):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device for metrics: {device}")

    try:
        niqe_metric = pyiqa.create_metric('niqe', device=device)
        lpips_metric = pyiqa.create_metric('lpips', device=device)
    except Exception as e:
        print(f"Failed to load PyIQA models: {e}")
        return None

    results = []
    if not os.path.exists(enhanced_dir):
        print(f"Directory not found: {enhanced_dir}")
        return None

    img_files = sorted([f for f in os.listdir(enhanced_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    print(f"Processing {len(img_files)} images...")

    for img_name in tqdm(img_files):
        enhanced_path = os.path.join(enhanced_dir, img_name)
        original_path = os.path.join(original_dir, img_name)

        try:
            img_dist = cv2.imread(enhanced_path)
            img_ref = cv2.imread(original_path)
            if img_dist is None: continue
            h, w = img_dist.shape[:2]
            min_dim = 64
            if h < min_dim or w < min_dim:
                img_dist = cv2.resize(img_dist, (max(w, min_dim), max(h, min_dim)))
            if img_ref is not None:
                h_r, w_r = img_ref.shape[:2]
                if h_r < min_dim or w_r < min_dim:
                    img_ref = cv2.resize(img_ref, (max(w_r, min_dim), max(h_r, min_dim)))
                if img_ref.shape != img_dist.shape:
                    img_ref = cv2.resize(img_ref, (img_dist.shape[1], img_dist.shape[0]))

            dist_tensor = torch.from_numpy(img_dist).float() / 255.0
            dist_tensor = dist_tensor.permute(2, 0, 1).unsqueeze(0).to(device)

            ref_tensor = None
            if img_ref is not None:
                ref_tensor = torch.from_numpy(img_ref).float() / 255.0
                ref_tensor = ref_tensor.permute(2, 0, 1).unsqueeze(0).to(device)
            score_niqe = np.nan
            score_lpips = np.nan

            with torch.no_grad():
                score_niqe = niqe_metric(dist_tensor).item()
                if ref_tensor is not None:
                    score_lpips = lpips_metric(dist_tensor, ref_tensor).item()

            results.append({
                'filename': img_name,
                'NIQE': score_niqe,
                'LPIPS_vs_Original': score_lpips
            })

        except Exception as e:
            continue
    if results:
        df = pd.DataFrame(results)
        os.makedirs(os.path.dirname(output_csv), exist_ok=True)
        df.to_csv(output_csv, index=False)
        return df
    else:
        print("No results computed.")
        return pd.DataFrame()

In [ ]:
base_path = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser"
original_dir = f"{base_path}/D_prime/image"

print("\n" + "="*70)
print("COMPUTING QUALITY METRICS FOR ALL METHODS")
print("="*70)

for method_name, dataset_name in method_configs.items():
    print(f"\n{'='*70}")
    print(f"Computing metrics for: {method_name}")
    print(f"{'='*70}")

    enhanced_dir = f"{base_path}/{dataset_name}/image"

    quality_csv = f"./result/None-VGG-None-CTC.pth/{dataset_name}_quality_scores.csv"

    if os.path.exists(quality_csv):
        print(f"ℹ️ Quality scores already exist: {quality_csv}")
    else:
        print(f"Computing sharp/noise/contrast/low_light scores...")
        df_quality = compute_quality_scores(enhanced_dir, quality_csv)
        print(f"✅ Quality scores saved")

    try:
        import torch
        niqe_lpips_csv = f"./result/None-VGG-None-CTC.pth/{dataset_name}_niqe_lpips.csv"

        if os.path.exists(niqe_lpips_csv):
            print(f"ℹ️ NIQE/LPIPS scores already exist: {niqe_lpips_csv}")
        else:
            print(f"Computing NIQE and LPIPS scores...")
            df_niqe_lpips = compute_quality_metrics_batch(original_dir, enhanced_dir, niqe_lpips_csv)
            print(f"✅ NIQE/LPIPS scores saved")
    except Exception as e:
        print(f"⚠️ Skipping NIQE/LPIPS (torch not available or error: {e})")

print("\n" + "="*70)
print("✅ ALL QUALITY METRICS COMPUTED")
print("="*70)


In [ ]:
import os
example_name = list(method_configs.values())[0]
file_path = f"./result/None-VGG-None-CTC.pth/{example_name}_niqe_lpips.csv"

print("save here")
print(os.path.abspath(file_path))

if os.path.exists(file_path):
    print("✅ document exist！")
else:
    print("doesn't have document")

In [ ]:
import glob
import os
import pandas as pd
df_blur = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_blur_levels.csv")
df_original = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_combined.csv")
df_original['method'] = 'Original'
all_combined_results = [df_original]

print("\n" + "="*70)
print("COMBINING ALL RESULTS (FIXED)")
print("="*70)

for method_name, dataset_name in method_configs.items():
    print(f"\nProcessing: {method_name}")
    result_pattern = f"./result/**/{dataset_name}_detailed_results.csv"
    ocr_files = glob.glob(result_pattern, recursive=True)

    if not ocr_files:
        print(f"⚠️ OCR results not found for {method_name}")
        continue
    ocr_csv = max(ocr_files, key=os.path.getmtime)
    print(f"  OCR results: {os.path.basename(ocr_csv)}")
    df_ocr = pd.read_csv(ocr_csv)
    if 'idx' not in df_ocr.columns:
        if 'image_path' in df_ocr.columns:
            try:
                df_ocr['idx'] = df_ocr['image_path'].apply(lambda x: int(os.path.basename(x).split('_')[0]))
            except:
                print("    ⚠️ Could not extract idx from image_path in OCR results")
        elif 'index' in df_ocr.columns:
             df_ocr['idx'] = df_ocr['index']
    quality_csv = f"./result/None-VGG-None-CTC.pth/{dataset_name}_quality_scores.csv"
    if os.path.exists(quality_csv):
        df_quality = pd.read_csv(quality_csv)
        print(f"  Quality scores: {os.path.basename(quality_csv)}")
    else:
        print(f"⚠️ Quality scores not found for {method_name}")
        df_quality = pd.DataFrame(columns=['idx'])
    niqe_lpips_csv = f"./result/None-VGG-None-CTC.pth/{dataset_name}_niqe_lpips.csv"
    if os.path.exists(niqe_lpips_csv):
        df_niqe_lpips = pd.read_csv(niqe_lpips_csv)
        print(f"  NIQE/LPIPS: {os.path.basename(niqe_lpips_csv)}")
        rename_map = {
            'NIQE': 'niqe_score',
            'LPIPS_vs_Original': 'lpips_score'
        }
        df_niqe_lpips.rename(columns=rename_map, inplace=True)
        if 'idx' not in df_niqe_lpips.columns and 'filename' in df_niqe_lpips.columns:
            try:
                df_niqe_lpips['idx'] = df_niqe_lpips['filename'].apply(lambda x: int(str(x).split('_')[0]))
            except Exception as e:
                print(f"    ⚠️ Failed to extract idx from filename in NIQE csv: {e}")
    else:
        df_niqe_lpips = None
        print(f"  NIQE/LPIPS: Not available")
    time_csv = f"./result/None-VGG-None-CTC.pth/{method_name}_processing_log.csv"
    if os.path.exists(time_csv):
        df_time = pd.read_csv(time_csv)
        print(f"  Processing time: {os.path.basename(time_csv)}")
        if 'idx' not in df_time.columns and 'filename' in df_time.columns:
             df_time['idx'] = df_time['filename'].apply(lambda x: int(str(x).split('_')[0]))
    else:
        df_time = None
    if not df_quality.empty and 'idx' in df_quality.columns:
        df_merged = df_ocr.merge(df_quality, on='idx', how='inner')
    else:
        df_merged = df_ocr
    if 'idx' in df_blur.columns:
        df_merged = df_merged.merge(df_blur[['idx', 'blur_variance', 'blur_level']], on='idx', how='inner')
    if df_niqe_lpips is not None and 'idx' in df_niqe_lpips.columns:
        cols_to_merge = ['idx', 'niqe_score', 'lpips_score']
        available_cols = [c for c in cols_to_merge if c in df_niqe_lpips.columns]
        df_merged = df_merged.merge(df_niqe_lpips[available_cols], on='idx', how='left')
    if df_time is not None and 'idx' in df_time.columns:
        df_merged = df_merged.merge(df_time[['idx', 'processing_time']], on='idx', how='left')
    else:
        df_merged['processing_time'] = None

    df_merged['method'] = method_name

    orig_idx = df_original.set_index('idx')

    if 'sharp_score' in df_merged.columns and 'sharp_score' in df_original.columns:
        df_merged['sharp_improvement'] = df_merged.apply(
            lambda row: row['sharp_score'] - orig_idx.loc[row['idx'], 'sharp_score'] if row['idx'] in orig_idx.index else 0,
            axis=1
        )

    # Save combined results for this method
    output_dir = "./result/None-VGG-None-CTC.pth"
    os.makedirs(output_dir, exist_ok=True)
    combined_csv = f"{output_dir}/{dataset_name}_combined.csv"
    df_merged.to_csv(combined_csv, index=False)
    print(f"  ✅ Saved: {os.path.basename(combined_csv)}")

    all_combined_results.append(df_merged)
if all_combined_results:
    df_comparison = pd.concat(all_combined_results, ignore_index=True)
    comparison_csv = "./result/None-VGG-None-CTC.pth/all_methods_comprehensive_comparison.csv"
    df_comparison.to_csv(comparison_csv, index=False)

    print("\n" + "="*70)
    print(f"✅ COMPREHENSIVE COMPARISON SAVED")
    print(f"File: {comparison_csv}")
    print(f"Total records: {len(df_comparison)}")
    print(f"Methods found: {df_comparison['method'].unique()}")
    print("="*70)
else:
    print("❌ No results collected.")


In [ ]:
usm_gt_path = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime_enhanced_USM/gt.txt"

if os.path.exists(usm_gt_path):
    print("修正 D_prime_enhanced_USM/gt.txt")
    with open(usm_gt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    corrected_lines = []
    for line in lines:
        line = line.strip()
        if line:
            parts = line.split('\t')
            if len(parts) == 2:
                image_path, label = parts
                if image_path.startswith('images/'):
                    image_path = image_path.replace('images/', '', 1)
                corrected_lines.append(f"{image_path}\t{label}\n")
    with open(usm_gt_path, 'w', encoding='utf-8') as f:
        f.writelines(corrected_lines)

    print(f"✅ fix already {usm_gt_path}")
else:
    print("D_prime_enhanced_USM/gt.txt not exist")

In [ ]:
import os
import shutil

print("="*70)
print("修正 gt.txt 文件（移除 images/ 前缀）")
print("="*70)
gt_path = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime/gt.txt"
backup_path = gt_path + ".backup"

if not os.path.exists(backup_path):
    shutil.copy(gt_path, backup_path)
    print(f"✅ save Again: {backup_path}")
else:
    print(f"document already exist: {backup_path}")
with open(gt_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"\n total {len(lines)} 行")
print(f"fix first: {lines[0].strip()}")

corrected_lines = []
corrected_count = 0

for line in lines:
    line = line.strip()
    if not line:
        continue

    parts = line.split('\t')
    if len(parts) == 2:
        image_path, label = parts
        if image_path.startswith('images/'):
            image_path = image_path.replace('images/', '', 1)
            corrected_count += 1

        corrected_lines.append(f"{image_path}\t{label}\n")
    else:
        print(f"skip wrong: {line}")

print(f"fix after {corrected_lines[0].strip()}")
print(f"total fix {corrected_count} ")

with open(gt_path, 'w', encoding='utf-8') as f:
    f.writelines(corrected_lines)

print(f"\n✅ save fix after gt.txt")

print("\n" + "-"*70)
print("check the result")
print("-"*70)
!head -5 {gt_path}

print("\n" + "-"*70)
print("check the path")
print("-"*70)

image_dir = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime/image"

with open(gt_path, 'r') as f:
    test_lines = [f.readline() for _ in range(5)]

errors = 0
for line in test_lines:
    img_path, label = line.strip().split('\t')
    full_path = os.path.join(image_dir, img_path)

    if os.path.exists(full_path):
        print(f"✅ {img_path}")
    else:
        print(f"{img_path} document not exist")
        errors += 1

if errors == 0:
    print(f"\nall true")
else:
    print(f"\n errors")

print("\n" + "="*70)
print("="*70)

In [ ]:
print("="*70)
print("="*70)

lmdb_dirs = [
    "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime_enhanced_USM/D_prime_enhanced_USM_lmdb",
]

for lmdb_dir in lmdb_dirs:
    if os.path.exists(lmdb_dir):
        import shutil
        shutil.rmtree(lmdb_dir)
        print(f"✅ deleted old ltmb {lmdb_dir}")
print("\nall problem fixed")

In [ ]:
import shutil

source_gt = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime/gt.txt"
target_gt = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime_enhanced_USM/gt.txt"

shutil.copy(source_gt, target_gt)
print(f"Copied gt.txt to: {target_gt}")

image_dir = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime_enhanced_USM/image"
gt_file = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime_enhanced_USM/gt.txt"
lmdb_output = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime_enhanced_USM/D_prime_enhanced_USM_lmdb"

print("\n" + "="*70)
print("Creating LMDB Dataset for D_prime_enhanced_USM")
print("="*70)

create_lmdb(image_dir, gt_file, lmdb_output)

print(f"\n LMDB created at: {lmdb_output}")


In [ ]:
print("\n" + "="*70)
print("Evaluating Model M on D'e_USM (USM Enhanced Dataset)")
print("="*70)

!python3 /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/deep-text-recognition-benchmark/test.py \
  --eval_data /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime_enhanced_USM/D_prime_enhanced_USM_lmdb \
  --data_filtering_off \
  --Transformation None --FeatureExtraction VGG --SequenceModeling None --Prediction CTC \
  --saved_model /content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/deep-text-recognition-benchmark/modules/None-VGG-None-CTC.pth

print("\nEvaluation completed. Results saved to ./result/...")

In [ ]:
print("\n" + "="*70)
print("Computing Quality Scores for USM Enhanced Images")
print("="*70)

df_usm_quality = compute_quality_scores(
    img_dir="/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime_enhanced_USM/image",
    out_csv="./result/None-VGG-None-CTC.pth/D_prime_enhanced_USM_quality_scores.csv"
)

print(f"\n✅ Quality scores computed for USM enhanced images")
print(f"Mean sharp_score: {df_usm_quality['sharp_score'].mean():.3f}")
print(f"Mean noise_score: {df_usm_quality['noise_score'].mean():.3f}")


In [ ]:
print("\n" + "="*70)
print("Combining OCR Results with Quality Scores")
print("="*70)

import glob
usm_result_files = glob.glob("./result/content_drive_MyDrive_UCSD_COURSES_ECE253_ImageCleanser_deep-text-recognition-benchmark_modules_None-VGG-None-CTC.pth/*USM*detailed_results.csv")

if usm_result_files:
    usm_result_csv = usm_result_files[0]
    print(f"Found USM result file: {usm_result_csv}")

    df_usm_ocr = pd.read_csv(usm_result_csv)
    df_usm_quality = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_enhanced_USM_quality_scores.csv")
    df_blur_levels = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_blur_levels.csv")

    df_usm_combined = df_usm_ocr.merge(df_usm_quality, on='idx', how='inner')
    df_usm_combined = df_usm_combined.merge(df_blur_levels[['idx', 'blur_variance', 'blur_level']], on='idx', how='inner')

    df_usm_combined.to_csv("./result/None-VGG-None-CTC.pth/D_prime_enhanced_USM_combined.csv", index=False)

    print(f"\n✅ Combined results saved to: ./result/None-VGG-None-CTC.pth/D_prime_enhanced_USM_combined.csv")
    print(f"Total records: {len(df_usm_combined)}")
else:
    print("USM result file not found")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("\n" + "="*70)
print("COMPARISON: R' (Original) vs R'e (USM Enhanced)")
print("="*70)

df_original = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_combined.csv")
df_original['version'] = 'Original (R\')'

try:
    df_usm = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_enhanced_USM_combined.csv")
    df_usm['version'] = 'USM Enhanced (R\'e)'

    df_compare = pd.concat([df_original, df_usm], ignore_index=True)

    print("\n" + "-"*60)
    print("Overall Performance Comparison")
    print("-"*60)

    summary = df_compare.groupby('version').agg({
        'confidence': ['mean', 'std'],
        'norm_edit_distance': ['mean', 'std']
    }).round(3)

    print(summary)

    acc_original = (df_original['norm_edit_distance'] < 1.0).mean() * 100
    acc_usm = (df_usm['norm_edit_distance'] < 1.0).mean() * 100

    print(f"\nAccuracy (NED < 1.0):")
    print(f"  Original: {acc_original:.1f}%")
    print(f"  USM:      {acc_usm:.1f}% (Δ = {acc_usm - acc_original:+.1f}%)")
    print("\n" + "-"*60)
    print("Performance by Blur Level")
    print("-"*60)

    blur_comparison = df_compare.groupby(['version', 'blur_level']).agg({
        'confidence': 'mean',
        'norm_edit_distance': 'mean'
    }).round(3)

    print(blur_comparison)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    sns.boxplot(data=df_compare, x='version', y='confidence', ax=axes[0,0], palette='Set2')
    axes[0,0].set_title('Confidence Distribution', fontsize=14, fontweight='bold')
    axes[0,0].set_ylabel('Confidence Score')
    axes[0,0].set_xlabel('')

    sns.boxplot(data=df_compare, x='version', y='norm_edit_distance', ax=axes[0,1], palette='Set2')
    axes[0,1].set_title('Normalized Edit Distance Distribution', fontsize=14, fontweight='bold')
    axes[0,1].set_ylabel('NED (lower is better)')
    axes[0,1].set_xlabel('')

    sns.boxplot(data=df_compare, x='blur_level', y='confidence', hue='version', ax=axes[1,0], palette='Set2')
    axes[1,0].set_title('Confidence by Blur Level', fontsize=14, fontweight='bold')
    axes[1,0].set_ylabel('Confidence Score')
    axes[1,0].set_xlabel('Blur Level')
    axes[1,0].legend(title='')

    blur_acc = df_compare.groupby(['version', 'blur_level']).apply(
        lambda x: (x['norm_edit_distance'] < 1.0).mean() * 100
    ).reset_index(name='accuracy')

    blur_pivot = blur_acc.pivot(index='blur_level', columns='version', values='accuracy')
    blur_pivot.plot(kind='bar', ax=axes[1,1], color=['#8dd3c7', '#fb8072'])
    axes[1,1].set_title('Accuracy by Blur Level', fontsize=14, fontweight='bold')
    axes[1,1].set_ylabel('Accuracy (%)')
    axes[1,1].set_xlabel('Blur Level')
    axes[1,1].legend(title='')
    axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=0)

    plt.tight_layout()
    plt.savefig('./result/None-VGG-None-CTC.pth/comparison_original_vs_USM.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\n✅ Comparison chart saved to: ./result/None-VGG-None-CTC.pth/comparison_original_vs_USM.png")
    from scipy import stats

    t_stat, p_value = stats.ttest_rel(
        df_original['confidence'].values,
        df_usm['confidence'].values
    )

    print("\n" + "-"*60)
    print("Statistical Significance Test (Paired t-test)")
    print("-"*60)
    print(f"Confidence improvement: {df_usm['confidence'].mean() - df_original['confidence'].mean():+.3f}")
    print(f"t-statistic: {t_stat:.3f}")
    print(f"p-value: {p_value:.4f}")

    if p_value < 0.001:
        print("Result: *** Highly significant (p < 0.001)")
    elif p_value < 0.01:
        print("Result: ** Very significant (p < 0.01)")
    elif p_value < 0.05:
        print("Result: * Significant (p < 0.05)")
    else:
        print("Result: Not significant (p >= 0.05)")

except FileNotFoundError as e:
    print(f"⚠️ File not found: {e}")
    print("Please run the evaluation first to generate results.")


In [ ]:
def visualize_usm_comparison(sample_indices=[50, 150, 250, 350, 450]):

    original_dir = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime/image"
    usm_dir = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser/D_prime_enhanced_USM/image"

    try:
        df_usm = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_enhanced_USM_combined.csv")
    except:
        print(" Combined results not found. Skipping visualization.")
        return

    original_files = sorted(os.listdir(original_dir))

    for idx in sample_indices:
        fname = [f for f in original_files if f.startswith(f"{idx}_")]
        if not fname:
            continue
        fname = fname[0]
        img_info = df_usm[df_usm['idx'] == idx].iloc[0]
        img_original = cv2.imread(os.path.join(original_dir, fname))
        img_usm = cv2.imread(os.path.join(usm_dir, fname))

        if img_original is None or img_usm is None:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        axes[0].imshow(cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Original (Blurred)', fontsize=14, fontweight='bold')
        axes[0].axis('off')
        axes[0].text(0.5, -0.05,
                    f"Blur: {img_info['blur_level']}\n"
                    f"Blur Variance: {img_info['blur_variance']:.1f}\n"
                    f"Ground Truth: {img_info['ground_truth']}",
                    transform=axes[0].transAxes,
                    ha='center', va='top',
                    fontsize=11,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        axes[1].imshow(cv2.cvtColor(img_usm, cv2.COLOR_BGR2RGB))
        axes[1].set_title('USM Enhanced', fontsize=14, fontweight='bold')
        axes[1].axis('off')
        axes[1].text(0.5, -0.05,
                    f"Prediction: {img_info['prediction']}\n"
                    f"Confidence: {img_info['confidence']:.3f}\n"
                    f"NED: {img_info['norm_edit_distance']:.3f}\n"
                    f"Sharp↑: {img_info['sharp_score']:.3f}",
                    transform=axes[1].transAxes,
                    ha='center', va='top',
                    fontsize=11,
                    bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

        plt.suptitle(f'Sample {idx}: {fname}', fontsize=16, fontweight='bold', y=0.98)
        plt.tight_layout()
        plt.savefig(f'./result/None-VGG-None-CTC.pth/usm_sample_{idx}.png', dpi=200, bbox_inches='tight')
        plt.show()

        print(f"✅ Saved: ./result/None-VGG-None-CTC.pth/usm_sample_{idx}.png\n")

print("\n" + "="*70)
print("Visualizing Sample Comparisons")
print("="*70)

visualize_usm_comparison(sample_indices=[50, 150, 250, 350, 450, 550])


In [ ]:
import pandas as pd

print("="*70)
print("check ")
print("="*70)

try:
    df_usm = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_enhanced_USM_combined.csv")
    print("\n✅ find D_prime_enhanced_USM_combined.csv")

    if 'blur_level' in df_usm.columns:
        print("\n✅ include blur_level ")
    else:
        print("\nlack of blur_level ")

except FileNotFoundError:
    print("\nnot found D_prime_enhanced_USM_combined.csv")
try:
    df_blur = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_blur_levels.csv")
    print("\nfind D_prime_blur_levels.csv")
except FileNotFoundError:
    print("\nnot found D_prime_blur_levels.csv")
try:
    df_orig = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_combined.csv")
    print("\n✅ found D_prime_combined.csv")
    if 'blur_level' in df_orig.columns:
        print("original data including blur_level ")
    else:
        print("lack of  blur_level ")
except FileNotFoundError:
    print("\nnot found D_prime_combined.csv")

In [ ]:
import pandas as pd

print("="*70)
print("fix blur_level")
print("="*70)

df_original = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_combined.csv")
print(f"\n original data: {len(df_original)} ")
print(f" {list(df_original.columns)}")
df_blur = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_blur_levels.csv")
print(f"\nblur level {len(df_blur)} ")
df_original_updated = df_original.merge(
    df_blur[['idx', 'blur_variance', 'blur_level']],
    on='idx',
    how='left'
)

print(f"\n after combine: {len(df_original_updated)}")
print(f"add new blur_variance, blur_level")

print(f"\n check blur_level :")
print(df_original_updated['blur_level'].value_counts())

df_original_updated.to_csv("./result/None-VGG-None-CTC.pth/D_prime_combined.csv", index=False)

print(f"\nupdate  D_prime_combined.csv")

print("\n" + "="*70)
print("fix finished")
print("="*70)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT SUMMARY: USM Deblurring Results")
print("="*70)

try:
    df_original = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_combined.csv")
    df_usm = pd.read_csv("./result/None-VGG-None-CTC.pth/D_prime_enhanced_USM_combined.csv")
    print("\n1. OVERALL PERFORMANCE")
    print("-" * 60)

    acc_orig = (df_original['norm_edit_distance'] < 1.0).mean() * 100
    acc_usm = (df_usm['norm_edit_distance'] < 1.0).mean() * 100
    conf_orig = df_original['confidence'].mean()
    conf_usm = df_usm['confidence'].mean()
    ned_orig = df_original['norm_edit_distance'].mean()
    ned_usm = df_usm['norm_edit_distance'].mean()

    print(f"{'Metric':<30} {'Original':>12} {'USM':>12} {'Improvement':>15}")
    print("-" * 60)
    print(f"{'Accuracy (%)':30} {acc_orig:12.1f} {acc_usm:12.1f} {acc_usm-acc_orig:+14.1f}")
    print(f"{'Confidence':30} {conf_orig:12.3f} {conf_usm:12.3f} {conf_usm-conf_orig:+14.3f}")
    print(f"{'NED (lower better)':30} {ned_orig:12.3f} {ned_usm:12.3f} {ned_usm-ned_orig:+14.3f}")
    print("\n2. PERFORMANCE BY BLUR LEVEL")
    print("-" * 60)
    print(f"{'Blur Level':<15} {'Original Acc':>15} {'USM Acc':>15} {'Improvement':>15}")
    print("-" * 60)

    for blur_level in ['light', 'moderate', 'heavy']:
        orig_subset = df_original[df_original['blur_level'] == blur_level]
        usm_subset = df_usm[df_usm['blur_level'] == blur_level]

        if len(orig_subset) > 0 and len(usm_subset) > 0:
            acc_orig_blur = (orig_subset['norm_edit_distance'] < 1.0).mean() * 100
            acc_usm_blur = (usm_subset['norm_edit_distance'] < 1.0).mean() * 100
            improvement = acc_usm_blur - acc_orig_blur

            print(f"{blur_level.capitalize():15} {acc_orig_blur:15.1f} {acc_usm_blur:15.1f} {improvement:+14.1f}")
    print("\n3. IMAGE QUALITY METRICS")
    print("-" * 60)

    sharp_orig = df_original['sharp_score'].mean()
    sharp_usm = df_usm['sharp_score'].mean()
    noise_orig = df_original['noise_score'].mean()
    noise_usm = df_usm['noise_score'].mean()

    print(f"{'Metric':<30} {'Original':>12} {'USM':>12} {'Change':>15}")
    print("-" * 60)
    print(f"{'Sharp Score (higher better)':30} {sharp_orig:12.3f} {sharp_usm:12.3f} {sharp_usm-sharp_orig:+14.3f}")
    print(f"{'Noise Score (lower better)':30} {noise_orig:12.3f} {noise_usm:12.3f} {noise_usm-noise_orig:+14.3f}")

    print("\n4. CONCLUSIONS")
    print("-" * 60)

    if acc_usm > acc_orig:
        print(f"✅ USM deblurring improves OCR accuracy by {acc_usm - acc_orig:.1f}%")
    else:
        print(f"❌ USM deblurring does not improve OCR accuracy")

    if sharp_usm > sharp_orig:
        print(f"✅ USM successfully increases image sharpness by {sharp_usm - sharp_orig:.3f}")

    if conf_usm > conf_orig:
        print(f"✅ USM increases model confidence by {conf_usm - conf_orig:.3f}")
    print("\n5. KEY FINDINGS")
    print("-" * 60)

    blur_dist = df_original['blur_level'].value_counts()
    print(f"• Dataset composition:")
    for level in ['light', 'moderate', 'heavy']:
        count = blur_dist.get(level, 0)
        pct = count / len(df_original) * 100
        print(f"  - {level.capitalize()}: {count} images ({pct:.1f}%)")
    improvements = []
    for blur_level in ['light', 'moderate', 'heavy']:
        orig_subset = df_original[df_original['blur_level'] == blur_level]
        usm_subset = df_usm[df_usm['blur_level'] == blur_level]

        if len(orig_subset) > 0:
            acc_orig_blur = (orig_subset['norm_edit_distance'] < 1.0).mean() * 100
            acc_usm_blur = (usm_subset['norm_edit_distance'] < 1.0).mean() * 100
            improvements.append((blur_level, acc_usm_blur - acc_orig_blur))

    best_improvement = max(improvements, key=lambda x: x[1])
    print(f"\n• Largest improvement: {best_improvement[0]} blur (+{best_improvement[1]:.1f}%)")

    print("\n" + "="*70)
    print("✅ EXPERIMENT COMPLETED SUCCESSFULLY")
    print("="*70)

    print("\nGenerated files:")
    print("  - D_prime_blur_levels.csv")
    print("  - D_prime_enhanced_USM_quality_scores.csv")
    print("  - D_prime_enhanced_USM_combined.csv")
    print("  - comparison_original_vs_USM.png")
    print("  - usm_sample_*.png")

except FileNotFoundError as e:
    print(f" Error: {e}")
    print("Please ensure all previous cells have been run successfully.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
csv_dprime = "./result/None-VGG-None-CTC.pth/D_prime_combined.csv"
csv_mj600  = "./result/None-VGG-None-CTC.pth/MJ600_combined.csv"
df_dprime = pd.read_csv(csv_dprime)
df_mj600  = pd.read_csv(csv_mj600)
df_dprime["dataset"] = "D_prime"
df_mj600["dataset"]  = "MJ600"
df_all = pd.concat([df_dprime, df_mj600], ignore_index=True)
performance_metrics = ["confidence", "norm_edit_distance"]
quality_metrics = ["sharp_score", "noise_score", "contrast_score", "low_light_score"]
quality_titles  = ["Sharpness", "Noise", "Contrast", "Low-light"]
sns.set(style="whitegrid")

for perf_metric in performance_metrics:
    for quality_metric, quality_title in zip(quality_metrics, quality_titles):
        plt.figure(figsize=(8, 6))
        sns.regplot(
            data=df_dprime,
            x=quality_metric,
            y=perf_metric,
            scatter_kws={'alpha':0.6, 's':15},
            label="D_prime",
            ci=95
        )
        sns.regplot(
            data=df_mj600,
            x=quality_metric,
            y=perf_metric,
            scatter_kws={'alpha':0.6, 's':15},
            label="MJ600",
            ci=95
        )
        plt.title(f"Correlation: {perf_metric.replace('_', ' ').title()} vs {quality_title}")
        plt.xlabel(f"{quality_title} Score (0-1)")
        plt.ylabel(perf_metric.replace('_', ' ').title())
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
!pip -q install lpips scipy pyiqa


In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import ndimage, signal
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings('ignore')

try:
    import lpips
    LPIPS_AVAILABLE = True
except:
    LPIPS_AVAILABLE = False
    print(" LPIPS not available, will skip LPIPS evaluation")

try:
    from pyiqa import create_metric
    NIQE_AVAILABLE = True
except:
    NIQE_AVAILABLE = False
    print(" pyiqa not available")

print("Libraries imported successfully")


### Implementation of Four Deblurring Methods


In [ ]:

def estimate_psf_motion(length=15, angle=0):

    psf = np.zeros((length, length))
    center = length // 2
    angle_rad = np.deg2rad(angle)
    for i in range(length):
        x = int(center + (i - center) * np.cos(angle_rad))
        y = int(center + (i - center) * np.sin(angle_rad))
        if 0 <= x < length and 0 <= y < length:
            psf[y, x] = 1.0
    psf = psf / (psf.sum() + 1e-8)
    return psf

def estimate_psf_defocus(radius=3):

    size = 2 * radius + 1
    psf = np.zeros((size, size))
    center = radius

    y, x = np.ogrid[:size, :size]
    mask = (x - center)**2 + (y - center)**2 <= radius**2
    psf[mask] = 1.0
    psf = psf / (psf.sum() + 1e-8)
    return psf


def apply_wiener_deconvolution(img, psf, noise_var=0.01):
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    img_float = img.astype(np.float32) / 255.0
    result_channels = []
    for c in range(img.shape[2]):
        channel = img_float[:, :, c]
        psf_padded = np.zeros_like(channel)
        psf_h, psf_w = psf.shape
        h_start = (channel.shape[0] - psf_h) // 2
        w_start = (channel.shape[1] - psf_w) // 2
        psf_padded[h_start:h_start+psf_h, w_start:w_start+psf_w] = psf
        img_fft = np.fft.fft2(channel)
        psf_fft = np.fft.fft2(np.fft.ifftshift(psf_padded))
        psf_conj = np.conj(psf_fft)
        psf_mag_sq = np.abs(psf_fft)**2
        wiener_filter = psf_conj / (psf_mag_sq + noise_var)
        restored_fft = img_fft * wiener_filter
        restored = np.real(np.fft.ifft2(restored_fft))
        restored = np.clip(restored, 0, 1)
        result_channels.append(restored)
    result = np.stack(result_channels, axis=2)
    result = (result * 255).astype(np.uint8)

    return result


def apply_richardson_lucy(img, psf, iterations=10):
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    img_float = img.astype(np.float32) / 255.0
    psf_float = psf.astype(np.float32)
    psf_flipped = np.flip(psf_float)
    result_channels = []
    for c in range(img.shape[2]):
        channel = img_float[:, :, c]
        estimate = channel.copy()
        psf_padded = np.zeros_like(channel)
        psf_h, psf_w = psf.shape
        h_start = (channel.shape[0] - psf_h) // 2
        w_start = (channel.shape[1] - psf_w) // 2
        psf_padded[h_start:h_start+psf_h, w_start:w_start+psf_w] = psf_float
        for i in range(iterations):
            blurred_estimate = signal.convolve2d(estimate, psf_padded, mode='same', boundary='symm')
            blurred_estimate = np.clip(blurred_estimate, 1e-10, 1.0)
            ratio = channel / blurred_estimate
            correction = signal.correlate2d(ratio, psf_flipped, mode='same', boundary='symm')
            estimate = estimate * correction
            estimate = np.clip(estimate, 0, 1)

        result_channels.append(estimate)
    result = np.stack(result_channels, axis=2)
    result = (result * 255).astype(np.uint8)

    return result


def estimate_blur_kernel_simple(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    angles = np.arctan2(gy, gx)
    hist, bins = np.histogram(angles.flatten(), bins=36)
    dominant_angle = np.degrees(bins[np.argmax(hist)])
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    variance = laplacian.var()

    if variance < 50:
        length = 15
    elif variance < 100:
        length = 10
    else:
        length = 5

    psf = estimate_psf_motion(length=int(length), angle=dominant_angle)
    return psf

def apply_blind_deblurring(img, max_iterations=5):
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    psf = estimate_blur_kernel_simple(img)
    current_img = img.copy()
    for i in range(max_iterations):
        current_img = apply_richardson_lucy(current_img, psf, iterations=3)
        if i < max_iterations - 1:
            psf = estimate_blur_kernel_simple(current_img)

    return current_img


print("✅ All deblurring methods implemented")


In [ ]:
print('✅ NIQE / LPIPS helpers are already initialized in Step 2.2 (see above).')


In [ ]:
def _get_blur_variance(img, blur_variance=None):
    if blur_variance is not None:
        return blur_variance
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    return float(laplacian.var())

def _choose_psf_from_blur(blur_variance):
    if blur_variance < 50:
        return estimate_psf_motion(length=15, angle=0)
    elif blur_variance < 100:
        return estimate_psf_motion(length=10, angle=0)
    else:
        return estimate_psf_defocus(radius=3)

def apply_deblurring_method(img, method='USM', **kwargs):
    start_time = time.time()

    if method == 'USM':
        k = kwargs.get('k', 1.5)
        sigma = kwargs.get('sigma', 1.0)
        result = apply_usm_deblur(img, k=k, sigma=sigma)

    elif method == 'Wiener':
        blur_variance = _get_blur_variance(img, kwargs.get('blur_variance', None))
        psf = _choose_psf_from_blur(blur_variance)
        noise_var = kwargs.get('noise_var', 0.01)
        result = apply_wiener_deconvolution(img, psf, noise_var=noise_var)

    elif method == 'RL':
        blur_variance = _get_blur_variance(img, kwargs.get('blur_variance', None))
        psf = _choose_psf_from_blur(blur_variance)
        iterations = kwargs.get('iterations', 10)
        result = apply_richardson_lucy(img, psf, iterations=iterations)

    elif method == 'Blind':
        max_iterations = kwargs.get('max_iterations', 5)
        result = apply_blind_deblurring(img, max_iterations=max_iterations)

    else:
        raise ValueError(f"Unknown method: {method}")

    processing_time = time.time() - start_time
    return result, processing_time


def process_dataset_with_all_methods(
    input_dir,
    output_base_dir,
    blur_levels_csv=None,
    methods=['USM', 'Wiener', 'RL', 'Blind']
):

    blur_levels_dict = {}
    if blur_levels_csv and os.path.exists(blur_levels_csv):
        df_blur = pd.read_csv(blur_levels_csv)
        for _, row in df_blur.iterrows():
            blur_levels_dict[int(row['idx'])] = {
                'blur_variance': row.get('blur_variance', None),
                'blur_level': row.get('blur_level', 'unknown')
            }
        print(f"✅ Loaded blur levels for {len(blur_levels_dict)} images")

    image_files = sorted([f for f in os.listdir(input_dir)
                          if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    print(f"\n{'='*70}")
    print(f"Processing {len(image_files)} images with {len(methods)} methods")
    print(f"{'='*70}")

    all_results = []
    original_metrics_cache = {}

    for method in methods:
        print(f"\n{'─'*70}")
        print(f"Method: {method}")
        print(f"{'─'*70}")
        method_output_dir = os.path.join(output_base_dir, f"D_prime_enhanced_{method}", "image")
        os.makedirs(method_output_dir, exist_ok=True)

        method_results = []

        for fname in tqdm(image_files, desc=f"Processing {method}"):
            img_path = os.path.join(input_dir, fname)
            img = cv2.imread(img_path)

            if img is None:
                continue
            try:
                idx = int(fname.split('_')[0])
            except:
                idx = -1
            blur_info = blur_levels_dict.get(idx, {})
            blur_variance = blur_info.get('blur_variance', None)
            try:
                deblurred, proc_time = apply_deblurring_method(
                    img, method=method, blur_variance=blur_variance
                )
            except Exception as e:
                print(f"⚠️ Error processing {fname} with {method}: {e}")
                continue
            out_path = os.path.join(method_output_dir, fname)
            cv2.imwrite(out_path, deblurred)
            if fname not in original_metrics_cache:
                original_metrics_cache[fname] = {
                    'niqe_original': compute_niqe_score(img),
                    'sharp_score_original': calculate_sharp_score(img),
                    'noise_score_original': calculate_noise_score(img),
                }

            niqe_original = original_metrics_cache[fname]['niqe_original']
            sharp_orig = original_metrics_cache[fname]['sharp_score_original']
            noise_orig = original_metrics_cache[fname]['noise_score_original']
            niqe_deblurred = compute_niqe_score(deblurred)
            lpips_score = compute_lpips_score(img, deblurred)

            sharp_deblur = calculate_sharp_score(deblurred)
            noise_deblur = calculate_noise_score(deblurred)

            method_results.append({
                'idx': idx,
                'filename': fname,
                'method': method,
                'processing_time': proc_time,
                'niqe_original': niqe_original,
                'niqe_deblurred': niqe_deblurred,
                'niqe_improvement': (niqe_original - niqe_deblurred) if (niqe_original is not None and niqe_deblurred is not None) else None,
                'lpips_score': lpips_score,
                'sharp_score_original': sharp_orig,
                'sharp_score_deblurred': sharp_deblur,
                'sharp_improvement': (sharp_deblur - sharp_orig) if (sharp_deblur is not None and sharp_orig is not None) else None,
                'noise_score_original': noise_orig,
                'noise_score_deblurred': noise_deblur,
                'noise_change': (noise_deblur - noise_orig) if (noise_deblur is not None and noise_orig is not None) else None,
                'blur_variance': blur_variance,
                'blur_level': blur_info.get('blur_level', 'unknown')
            })

        all_results.extend(method_results)
        df_method = pd.DataFrame(method_results)
        method_csv = os.path.join(output_base_dir, f"D_prime_enhanced_{method}_metrics.csv")
        df_method.to_csv(method_csv, index=False)
        print(f"✅ Saved metrics: {method_csv}")

    # Combine all results
    results_df = pd.DataFrame(all_results)
    combined_csv = os.path.join(output_base_dir, "all_deblurring_methods_metrics.csv")
    results_df.to_csv(combined_csv, index=False)
    print(f"\n✅ Combined metrics saved: {combined_csv}")

    return results_df


print("✅ Complete processing pipeline ready")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

comparison_csv = "./result/None-VGG-None-CTC.pth/all_methods_comprehensive_comparison.csv"
if os.path.exists(comparison_csv):
    df_comparison = pd.read_csv(comparison_csv)
else:
    print("⚠️ Comparison CSV not found. Please run previous cells first.")
    df_comparison = None

if df_comparison is not None:
    sns.set_style("whitegrid")
    plt.rcParams['figure.figsize'] = (14, 10)

    #  OCR Performance Comparison
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    #  Accuracy by Method
    methods_order = ['Original', 'USM', 'Wiener', 'RL', 'Blind']
    accuracy_data = []
    for method in methods_order:
        if method in df_comparison['method'].values:
            df_method = df_comparison[df_comparison['method'] == method]
            acc = (df_method['norm_edit_distance'] < 1.0).mean() * 100
            accuracy_data.append({'method': method, 'accuracy': acc})

    df_acc = pd.DataFrame(accuracy_data)
    sns.barplot(data=df_acc, x='method', y='accuracy', ax=axes[0, 0], palette='viridis')
    axes[0, 0].set_title('OCR Accuracy by Method (NED < 1.0)', fontsize=14, fontweight='bold')
    axes[0, 0].set_ylabel('Accuracy (%)')
    axes[0, 0].set_xlabel('Method')

    sns.boxplot(data=df_comparison, x='method', y='confidence', ax=axes[0, 1], order=methods_order, palette='Set2')
    axes[0, 1].set_title('Confidence Score Distribution', fontsize=14, fontweight='bold')
    axes[0, 1].set_ylabel('Confidence')
    axes[0, 1].set_xlabel('Method')
    axes[0, 1].tick_params(axis='x', rotation=45)

    sns.boxplot(data=df_comparison, x='method', y='norm_edit_distance', ax=axes[0, 2], order=methods_order, palette='Set2')
    axes[0, 2].set_title('Normalized Edit Distance Distribution', fontsize=14, fontweight='bold')
    axes[0, 2].set_ylabel('NED (lower is better)')
    axes[0, 2].set_xlabel('Method')
    axes[0, 2].tick_params(axis='x', rotation=45)

    time_data = df_comparison.groupby('method')['processing_time'].mean().reset_index()
    time_data = time_data[time_data['method'].isin(methods_order)]
    sns.barplot(data=time_data, x='method', y='processing_time', ax=axes[1, 0], order=methods_order, palette='coolwarm')
    axes[1, 0].set_title('Average Processing Time per Image', fontsize=14, fontweight='bold')
    axes[1, 0].set_ylabel('Time (seconds)')
    axes[1, 0].set_xlabel('Method')
    axes[1, 0].tick_params(axis='x', rotation=45)

    niqe_data = df_comparison[df_comparison['method'] != 'Original'].copy()
    if 'niqe_improvement' in niqe_data.columns:
        sns.boxplot(data=niqe_data, x='method', y='niqe_improvement', ax=axes[1, 1], palette='Set2')
        axes[1, 1].set_title('NIQE Improvement (Lower is Better)', fontsize=14, fontweight='bold')
        axes[1, 1].set_ylabel('NIQE Improvement')
        axes[1, 1].set_xlabel('Method')
        axes[1, 1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
        axes[1, 1].tick_params(axis='x', rotation=45)

    sharp_data = df_comparison[df_comparison['method'] != 'Original'].copy()
    if 'sharp_improvement' in sharp_data.columns:
        sns.boxplot(data=sharp_data, x='method', y='sharp_improvement', ax=axes[1, 2], palette='Set2')
        axes[1, 2].set_title('Sharpness Score Improvement', fontsize=14, fontweight='bold')
        axes[1, 2].set_ylabel('Sharpness Improvement')
        axes[1, 2].set_xlabel('Method')
        axes[1, 2].axhline(y=0, color='r', linestyle='--', alpha=0.5)
        axes[1, 2].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig('./result/None-VGG-None-CTC.pth/comprehensive_methods_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("✅ Comprehensive comparison chart saved")


In [ ]:
if df_comparison is not None and 'blur_level' in df_comparison.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    blur_acc_data = []
    for method in methods_order:
        if method in df_comparison['method'].values:
            for blur_level in ['light', 'moderate', 'heavy']:
                df_subset = df_comparison[(df_comparison['method'] == method) &
                                         (df_comparison['blur_level'] == blur_level)]
                if len(df_subset) > 0:
                    acc = (df_subset['norm_edit_distance'] < 1.0).mean() * 100
                    blur_acc_data.append({
                        'method': method,
                        'blur_level': blur_level,
                        'accuracy': acc
                    })

    df_blur_acc = pd.DataFrame(blur_acc_data)
    if len(df_blur_acc) > 0:
        blur_pivot = df_blur_acc.pivot(index='blur_level', columns='method', values='accuracy')
        blur_pivot.plot(kind='bar', ax=axes[0, 0], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
        axes[0, 0].set_title('Accuracy by Blur Level', fontsize=14, fontweight='bold')
        axes[0, 0].set_ylabel('Accuracy (%)')
        axes[0, 0].set_xlabel('Blur Level')
        axes[0, 0].legend(title='Method')
        axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=0)

    blur_ned = df_comparison.groupby(['method', 'blur_level'])['norm_edit_distance'].mean().reset_index()
    blur_ned_pivot = blur_ned.pivot(index='blur_level', columns='method', values='norm_edit_distance')
    blur_ned_pivot.plot(kind='bar', ax=axes[0, 1], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
    axes[0, 1].set_title('Normalized Edit Distance by Blur Level', fontsize=14, fontweight='bold')
    axes[0, 1].set_ylabel('NED (lower is better)')
    axes[0, 1].set_xlabel('Blur Level')
    axes[0, 1].legend(title='Method')
    axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)

    blur_conf = df_comparison.groupby(['method', 'blur_level'])['confidence'].mean().reset_index()
    blur_conf_pivot = blur_conf.pivot(index='blur_level', columns='method', values='confidence')
    blur_conf_pivot.plot(kind='bar', ax=axes[1, 0], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
    axes[1, 0].set_title('Confidence by Blur Level', fontsize=14, fontweight='bold')
    axes[1, 0].set_ylabel('Confidence')
    axes[1, 0].set_xlabel('Blur Level')
    axes[1, 0].legend(title='Method')
    axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=0)

    time_by_method = df_comparison.groupby('method')['processing_time'].agg(['mean', 'std']).reset_index()
    time_by_method = time_by_method[time_by_method['method'].isin(methods_order)]
    axes[1, 1].bar(time_by_method['method'], time_by_method['mean'],
                   yerr=time_by_method['std'], capsize=5, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
    axes[1, 1].set_title('Processing Time (Mean ± Std)', fontsize=14, fontweight='bold')
    axes[1, 1].set_ylabel('Time (seconds)')
    axes[1, 1].set_xlabel('Method')
    axes[1, 1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig('./result/None-VGG-None-CTC.pth/performance_by_blur_level.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("✅ Blur level analysis chart saved")


In [ ]:
if df_comparison is not None:
    print("\n" + "="*70)
    print("FINAL EXPERIMENT SUMMARY: Four Deblurring Methods")
    print("="*70)

    # Overall performance
    print("\n1. OVERALL OCR PERFORMANCE")
    print("-" * 70)
    print(f"{'Method':<15} {'Accuracy (%)':>15} {'Confidence':>15} {'NED':>15} {'Time (s)':>15}")
    print("-" * 70)

    for method in methods_order:
        if method in df_comparison['method'].values:
            df_method = df_comparison[df_comparison['method'] == method]
            acc = (df_method['norm_edit_distance'] < 1.0).mean() * 100
            conf = df_method['confidence'].mean()
            ned = df_method['norm_edit_distance'].mean()
            if method != 'Original' and 'processing_time' in df_method.columns:
                time_avg = df_method['processing_time'].mean()
            else:
                time_avg = 0.0
            print(f"{method:<15} {acc:>15.2f} {conf:>15.3f} {ned:>15.3f} {time_avg:>15.4f}")

    print("\n2. PERFORMANCE BY BLUR LEVEL")
    print("-" * 70)
    for blur_level in ['light', 'moderate', 'heavy']:
        print(f"\n{blur_level.upper()} Blur (σ² {['>100', '50-100', '<50'][['light', 'moderate', 'heavy'].index(blur_level)]}):")
        print(f"{'Method':<15} {'Accuracy (%)':>15} {'Confidence':>15} {'NED':>15}")
        print("-" * 60)

        for method in methods_order:
            if method in df_comparison['method'].values:
                df_subset = df_comparison[(df_comparison['method'] == method) &
                                         (df_comparison['blur_level'] == blur_level)]
                if len(df_subset) > 0:
                    acc = (df_subset['norm_edit_distance'] < 1.0).mean() * 100
                    conf = df_subset['confidence'].mean()
                    ned = df_subset['norm_edit_distance'].mean()
                    print(f"{method:<15} {acc:>15.2f} {conf:>15.3f} {ned:>15.3f}")

    # Quality metrics
    print("\n3. IMAGE QUALITY METRICS")
    print("-" * 70)
    quality_methods = [m for m in methods_order if m != 'Original']
    if 'niqe_improvement' in df_comparison.columns:
        print(f"{'Method':<15} {'NIQE Improvement':>20} {'Sharp Improvement':>20} {'Noise Change':>20}")
        print("-" * 75)
        for method in quality_methods:
            if method in df_comparison['method'].values:
                df_method = df_comparison[df_comparison['method'] == method]
                niqe_imp = df_method['niqe_improvement'].mean() if 'niqe_improvement' in df_method.columns else None
                sharp_imp = df_method['sharp_improvement'].mean() if 'sharp_improvement' in df_method.columns else None
                noise_chg = df_method['noise_change'].mean() if 'noise_change' in df_method.columns else None

                niqe_str = f"{niqe_imp:.3f}" if niqe_imp is not None else "N/A"
                sharp_str = f"{sharp_imp:.3f}" if sharp_imp is not None else "N/A"
                noise_str = f"{noise_chg:.3f}" if noise_chg is not None else "N/A"

                print(f"{method:<15} {niqe_str:>20} {sharp_str:>20} {noise_str:>20}")

    print("\n4. KEY FINDINGS")
    print("-" * 70)

    for blur_level in ['light', 'moderate', 'heavy']:
        best_method = None
        best_acc = 0
        for method in quality_methods:
            if method in df_comparison['method'].values:
                df_subset = df_comparison[(df_comparison['method'] == method) &
                                         (df_comparison['blur_level'] == blur_level)]
                if len(df_subset) > 0:
                    acc = (df_subset['norm_edit_distance'] < 1.0).mean() * 100
                    if acc > best_acc:
                        best_acc = acc
                        best_method = method
        if best_method:
            print(f"• Best method for {blur_level} blur: {best_method} ({best_acc:.1f}% accuracy)")

    if 'processing_time' in df_comparison.columns:
        time_data = df_comparison[df_comparison['method'] != 'Original'].groupby('method')['processing_time'].mean()
        fastest = time_data.idxmin()
        fastest_time = time_data.min()
        print(f"• Fastest method: {fastest} ({fastest_time:.4f}s per image)")

    orig_acc = (df_comparison[df_comparison['method'] == 'Original']['norm_edit_distance'] < 1.0).mean() * 100
    best_improvement = 0
    best_method_imp = None
    for method in quality_methods:
        if method in df_comparison['method'].values:
            df_method = df_comparison[df_comparison['method'] == method]
            acc = (df_method['norm_edit_distance'] < 1.0).mean() * 100
            improvement = acc - orig_acc
            if improvement > best_improvement:
                best_improvement = improvement
                best_method_imp = method

    if best_method_imp:
        print(f"• Largest accuracy improvement: {best_method_imp} (+{best_improvement:.1f}% over original)")

    print("\n" + "="*70)
    print("✅ EXPERIMENT COMPLETE")
    print("="*70)

    print("\nGenerated files:")
    print("  - all_deblurring_methods_metrics.csv")
    print("  - all_methods_comprehensive_comparison.csv")
    print("  - comprehensive_methods_comparison.png")
    print("  - performance_by_blur_level.png")
    print("  - D_prime_enhanced_*_combined.csv (for each method)")
else:
    print("⚠️ Comparison data not available. ")


In [ ]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt

def visualize_all_methods_comparison_5cases(sample_indices=[50, 150, 250, 350, 450]):

    repo_root = "/content/drive/MyDrive/UCSD_COURSES/ECE253/ImageCleanser/repos/ImageCleanser"

    paths = {
        'Original': os.path.join(repo_root, "D_prime/image"),
        'USM':      os.path.join(repo_root, "D_prime_enhanced_USM/image"),
        'RL':       os.path.join(repo_root, "D_prime_enhanced_RL/image"),
        'Wiener':   os.path.join(repo_root, "D_prime_enhanced_Wiener/image"),
        'Blind':    os.path.join(repo_root, "D_prime_enhanced_Blind/image")
    }

    csvs = {
        'USM':    "./result/None-VGG-None-CTC.pth/D_prime_enhanced_USM_combined.csv",
        'RL':     "./result/None-VGG-None-CTC.pth/D_prime_enhanced_RL_combined.csv",
        'Wiener': "./result/None-VGG-None-CTC.pth/D_prime_enhanced_Wiener_combined.csv",
        'Blind':  "./result/None-VGG-None-CTC.pth/D_prime_enhanced_Blind_combined.csv"  # add
    }

    dfs = {}
    try:

        dfs['Original'] = pd.read_csv(csvs['USM'])

        for name, path in csvs.items():
            if os.path.exists(path):
                dfs[name] = pd.read_csv(path)
            else:
                print(f"⚠️ Warning: CSV for {name} not found: {path}")
                return

        print("✅ Loaded all data tables.")
    except Exception as e:
        print(f"Error loading CSVs: {e}")
        return

    if not os.path.exists(paths['Original']):
        print("Original path not found.")
        return

    original_files = sorted(os.listdir(paths['Original']))
    print(f"\nGenerating 5-Cases Grid for indices: {sample_indices}")

    method_order = ['Original', 'USM', 'RL', 'Wiener', 'Blind']
    colors = {'USM': '#2980b9', 'RL': '#c0392b', 'Wiener': '#27ae60', 'Blind': '#8e44ad'}
    titles = {
        'Original': 'Original (Blurred)',
        'USM': 'USM (Sharpening)',
        'RL': 'RL (Iterative)',
        'Wiener': 'Wiener (Best)',
        'Blind': 'Blind (Kernel Est.)'
    }
    notes = {
        'USM': "Edges harder but structure blurred",
        'RL': "Structure recovered but NOISY",
        'Wiener': "Balanced & Clean (Highest Acc)",
        'Blind': "Kernel-estimated deblur (Aggressive)"
    }

    nrows = len(sample_indices)
    ncols = len(method_order)

    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 5.0 * nrows))
    fig.patch.set_facecolor('#f4f6f7')

    if nrows == 1:
        axes = [axes]

    for r, idx in enumerate(sample_indices):
        fname_candidates = [f for f in original_files if f.startswith(f"{idx}_")]
        if not fname_candidates:
            for c in range(ncols):
                ax = axes[r][c]
                ax.axis('off')
                ax.text(0.5, 0.5, f"idx {idx} not found", ha='center', va='center', fontsize=14)
            continue

        fname = fname_candidates[0]

        row_orig = dfs['Original'][dfs['Original']['idx'] == idx]
        gt = row_orig.iloc[0]['ground_truth'] if not row_orig.empty else "N/A"

        for c, m in enumerate(method_order):
            ax = axes[r][c]
            ax.axis('off')

            if r == 0:
                if m == 'Original':
                    ax.set_title(f"{c+1}. {titles[m]}", fontsize=18, fontweight='bold', color='#555')
                else:
                    ax.set_title(f"{c+1}. {titles[m]}", fontsize=18, fontweight='bold', color=colors[m])

            img_path = os.path.join(paths[m], fname)
            img = cv2.imread(img_path) if os.path.exists(img_path) else None

            if img is None:
                ax.text(0.5, 0.5, "Image Not Found", ha='center', va='center', fontsize=14)
                continue

            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

            if m == 'Original':
                ax.text(
                    0.5, -0.18,
                    f"Sample #{idx}\nGT: {gt}",
                    transform=ax.transAxes, ha='center', fontsize=12,
                    bbox=dict(boxstyle='round,pad=0.5', facecolor='#bdc3c7', edgecolor='none')
                )
            else:
                row = dfs[m][dfs[m]['idx'] == idx]
                if row.empty:
                    pred, conf, ned = "N/A", 0.0, 1.0
                    is_correct = False
                else:
                    item = row.iloc[0]
                    pred = str(item.get('prediction', ''))
                    conf = float(item.get('confidence', 0.0))
                    ned  = float(item.get('norm_edit_distance', 1.0))
                    is_correct = (pred == str(gt))

                bg = '#d5f5e3' if is_correct else '#fadbd8'
                note = notes.get(m, "")

                ax.text(
                    0.5, -0.18,
                    f"Pred: {pred}\nConf: {conf:.3f} | NED: {ned:.2f}\n{note}",
                    transform=ax.transAxes, ha='center', fontsize=11,
                    bbox=dict(boxstyle='round,pad=0.5', facecolor=bg, edgecolor='none')
                )

    plt.suptitle("Full Evolution Analysis: 5 Samples × (Original + 4 Methods)", fontsize=26, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.98])

    save_path = "./result/None-VGG-None-CTC.pth/full_comparison_5cases_with_blind.png"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved full comparison grid: {save_path}")


visualize_all_methods_comparison_5cases(sample_indices=[50, 150, 250, 350, 450])
